In [6]:
from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph,START,END

load_dotenv(override = True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明状态
class OverAllState(MessagesState):
    output:str

#2. 声明节点
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_mode)
builder.add_node("output_node",output_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)

#4. 配置检查点存储器
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

#5. 使用的时候必须填写线程ID
config = {
    "configurable":{
        "thread_id":"chapter03-01"
    }
}

#6. 执行图
graph.invoke({"messages":[HumanMessage("你好,我是老王")]},config=config)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='85c2a32f-54c7-49da-8743-32c1ef62f171'),
  AIMessage(content='老王你好！我是DeepSeek，很高兴为你服务。有什么我可以帮你的吗？无论是生活琐事、工作问题，还是想聊聊天，我都随时在线。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 8, 'total_tokens': 45, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b23e50ff-b08e-4b2a-a288-52023900421a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f36e0-1305-7f81-8818-f27d6e5e788f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 37, 'total_tokens': 45, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': '

In [7]:
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='85c2a32f-54c7-49da-8743-32c1ef62f171'),
  AIMessage(content='老王你好！我是DeepSeek，很高兴为你服务。有什么我可以帮你的吗？无论是生活琐事、工作问题，还是想聊聊天，我都随时在线。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 8, 'total_tokens': 45, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b23e50ff-b08e-4b2a-a288-52023900421a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f36e0-1305-7f81-8818-f27d6e5e788f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 37, 'total_tokens': 45, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessag

In [8]:
config1 = {
    "configurable":{
        "thread_id":"chapter03-01xx"
    }
}
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config1)


{'messages': [HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='96a53e76-4acf-440b-99c2-2d4a76baa29e'),
  AIMessage(content='你好！很高兴见到你！不过，由于我们刚刚开始对话，我暂时还不知道你的身份信息呢。你可以告诉我你的名字或者任何你想让我知道的称呼，这样我就能更好地与你交流啦！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 8, 'total_tokens': 51, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2e946c0a-8ff9-4020-9d49-46af2a7e34dd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f36e1-d56e-7033-8153-98ab290048b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 43, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]